In [1]:
import pandas as pd
from pathlib import Path

# Menentukan folder proyek
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_path = project_root / "data" / "raw" / "Dataset Capstone.xlsx"
processed_dir = project_root / "data" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

# Membaca dataset menu
df_menu = pd.read_excel(
    raw_path,
    sheet_name="menu"
)

# Membuat salinan untuk cleaning
df_clean = df_menu.copy()

print("Jumlah data awal:", len(df_clean))
print("Jumlah kolom:", len(df_clean.columns))

display(df_clean.head())

Jumlah data awal: 183
Jumlah kolom: 15


,menu_id,branch_id,menu_name,category,description,price,currency,image_url,is_best_seller,is_available,source_id,source_url,accessed_at,verification_status,data_type
0,M001,BR001,Orange BBQ,Wings Platter,Hidangan wings platter dengan varian saus Oran...,35500,IDR,https://drive.google.com/file/d/1a2plPnYBaK482...,NaN,NaN,"Instagram, Gopay bio",https://www.instagram.com/kopilogi?dlrf=MWVpcD...,2026-11-10,source_checked,actual
1,M002,BR001,Garlic Parmesan,Wings Platter,Hidangan wings platter dengan perpaduan cita r...,35500,IDR,https://drive.google.com/file/d/1a2plPnYBaK482...,NaN,NaN,"Instagram, Gopay bio",https://www.instagram.com/kopilogi?dlrf=MWVpcD...,2026-11-10,source_checked,actual
2,M003,BR001,Salted Egg,Wings Platter,Hidangan wings platter dengan saus bercita ras...,35500,IDR,https://drive.google.com/file/d/1a2plPnYBaK482...,NaN,NaN,"Instagram, Gopay bio",https://www.instagram.com/kopilogi?dlrf=MWVpcD...,2026-11-10,source_checked,actual
3,M004,BR001,Korean Sauce,Wings Platter,Hidangan wings platter dengan pilihan saus ber...,35500,IDR,https://drive.google.com/file/d/1a2plPnYBaK482...,NaN,NaN,"Instagram, Gopay bio",https://www.instagram.com/kopilogi?dlrf=MWVpcD...,2026-11-10,source_checked,actual
4,M005,BR001,Hot Peanut,Pasta,Hidangan pasta dengan perpaduan cita rasa kaca...,23500,IDR,https://drive.google.com/file/d/1a2plPnYBaK482...,NaN,NaN,"Instagram, Gopay bio",https://www.instagram.com/kopilogi?dlrf=MWVpcD...,2026-11-10,source_checked,actual


In [2]:
# Menstandarkan nama kolom
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print("Nama kolom setelah standardisasi:")
print(df_clean.columns.tolist())

Nama kolom setelah standardisasi:
['menu_id', 'branch_id', 'menu_name', 'category', 'description', 'price', 'currency', 'image_url', 'is_best_seller', 'is_available', 'source_id', 'source_url', 'accessed_at', 'verification_status', 'data_type']


In [3]:
text_columns = [
    "menu_id",
    "branch_id",
    "menu_name",
    "category",
    "description",
    "currency",
    "image_url",
    "source_id",
    "source_url",
    "verification_status",
    "data_type"
]

for col in text_columns:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
    )

print("Standardisasi spasi selesai.")

display(
    df_clean[
        ["menu_id", "menu_name", "category", "price"]
    ].head()
)

Standardisasi spasi selesai.


,menu_id,menu_name,category,price
0,M001,Orange BBQ,Wings Platter,35500
1,M002,Garlic Parmesan,Wings Platter,35500
2,M003,Salted Egg,Wings Platter,35500
3,M004,Korean Sauce,Wings Platter,35500
4,M005,Hot Peanut,Pasta,23500


In [4]:
# Memastikan harga berupa angka
df_clean["price"] = pd.to_numeric(
    df_clean["price"],
    errors="coerce"
)

print("Tipe data harga:", df_clean["price"].dtype)

print("Jumlah harga kosong:", df_clean["price"].isna().sum())

print("Jumlah harga negatif:", (df_clean["price"] < 0).sum())

print("Jumlah harga nol:", (df_clean["price"] == 0).sum())

Tipe data harga: int64
Jumlah harga kosong: 0
Jumlah harga negatif: 0
Jumlah harga nol: 0


In [5]:
# Mempertahankan nilai kosong sebagai unknown
df_clean["is_best_seller"] = (
    df_clean["is_best_seller"].astype("boolean")
)

df_clean["is_available"] = (
    df_clean["is_available"].astype("boolean")
)

print("Missing values setelah penyesuaian tipe data:")

display(
    df_clean[
        ["is_best_seller", "is_available"]
    ].isna().sum()
)

Missing values setelah penyesuaian tipe data:


is_best_seller    183
is_available      183
dtype: int64

In [6]:
category_mapping = {
    "Bites & Fries": "food",
    "Nasi": "food",
    "Pasta": "food",
    "Wings Platter": "food",
    "Indomie Series": "food",

    "Freefrapp": "beverage",
    "Opieun": "beverage",
    "Es Kopilogi": "beverage",
    "Mocktail Shaken": "beverage",
    "Base Espresso": "beverage",
    "Minuman (Koplog)": "beverage",
    "Non/Coffee & Tonic": "beverage",
    "Kopipora": "beverage",
    "Bottled Coffee": "beverage",
    "Minuman (Teh)": "beverage",
    "Minuman (Limun)": "beverage",
    "Minuman (Non Kopi)": "beverage",

    "Sweet Treats": "dessert"
}

df_clean["category_group"] = (
    df_clean["category"].map(category_mapping)
)

print("Kategori yang belum dipetakan:")

display(
    df_clean.loc[
        df_clean["category_group"].isna(),
        "category"
    ].value_counts()
)

Kategori yang belum dipetakan:


category
Ngopay (Ngopi bari mayan)    8
Ngopay (Paket)               2
Name: count, dtype: int64[pyarrow]

In [7]:
print("=== VALIDASI DATA CLEANING ===")

print("Jumlah data awal:", len(df_menu))
print("Jumlah data setelah cleaning:", len(df_clean))

print(
    "Duplikasi menu_id:",
    df_clean["menu_id"].duplicated().sum()
)

print(
    "Harga negatif:",
    (df_clean["price"] < 0).sum()
)

print(
    "Kategori asli:",
    df_clean["category"].nunique()
)

print(
    "Kategori umum:",
    df_clean["category_group"].value_counts(dropna=False)
)

assert len(df_menu) == len(df_clean), \
    "Jumlah record berubah!"

assert df_clean["menu_id"].is_unique, \
    "Terdapat menu_id duplikat!"

assert not (df_clean["price"] < 0).any(), \
    "Terdapat harga negatif!"

print("\nValidasi dasar berhasil.")

=== VALIDASI DATA CLEANING ===
Jumlah data awal: 183
Jumlah data setelah cleaning: 183
Duplikasi menu_id: 0
Harga negatif: 0
Kategori asli: 20
Kategori umum: category_group
beverage    102
food         65
NaN          10
dessert       6
Name: count, dtype: int64

Validasi dasar berhasil.


In [9]:
print("=== PEMERIKSAAN MENU NGOPAY ===")

ngopay = df_clean[
    df_clean["category"].isin([
        "Ngopay (Ngopi bari mayan)",
        "Ngopay (Paket)"
    ])
]

display(
    ngopay[
        ["menu_id", "menu_name", "category", "price", "description"]
    ]
)

=== PEMERIKSAAN MENU NGOPAY ===


,menu_id,menu_name,category,price,description
117,M118,KOPLOG HITAM,Ngopay (Ngopi bari mayan),10000,Minuman Koplog varian hitam dengan karakter ra...
118,M119,KOPLOG SUSU,Ngopay (Ngopi bari mayan),10000,Minuman Koplog dengan perpaduan kopi dan susu ...
119,M120,KOPLOG GREENTEA,Ngopay (Ngopi bari mayan),12000,Minuman Koplog dengan cita rasa green tea dala...
120,M121,KOPLOG LEMON,Ngopay (Ngopi bari mayan),12000,Minuman Koplog dengan varian rasa lemon dalam ...
121,M122,KOPLOG SUSU BUTTER,Ngopay (Ngopi bari mayan),13000,Minuman Koplog dengan perpaduan susu dan butte...
122,M123,KOPLOG SUSU TELOR,Ngopay (Ngopi bari mayan),13000,Minuman Koplog dengan perpaduan susu dan telur...
123,M124,KOPLOG SUSU JELLY,Ngopay (Ngopi bari mayan),12000,Minuman Koplog dengan perpaduan susu dan jelly...
124,M125,MANUAL BREW,Ngopay (Ngopi bari mayan),13000,Minuman kopi yang diseduh menggunakan metode m...
125,M126,KOPLOG SUSU SAJODO 2 PCS JAJANAN PASAR,Ngopay (Paket),15000,Paket minuman Koplog susu yang dipadukan denga...
126,M127,KOPLOG HITAM SAJODO BALA-BALA,Ngopay (Paket),20000,Paket minuman Koplog hitam yang dipadukan deng...


In [10]:
# Menambahkan pemetaan kategori Ngopay
category_mapping.update({
    "Ngopay (Ngopi bari mayan)": "beverage",
    "Ngopay (Paket)": "package"
})

# Memperbarui kategori umum
df_clean["category_group"] = (
    df_clean["category"].map(category_mapping)
)

print("=== HASIL KATEGORI UMUM ===")

display(
    df_clean["category_group"].value_counts(
        dropna=False
    )
)

print(
    "Kategori belum dipetakan:",
    df_clean["category_group"].isna().sum()
)

=== HASIL KATEGORI UMUM ===


category_group
beverage    110
food         65
dessert       6
package       2
Name: count, dtype: int64

Kategori belum dipetakan: 0


In [11]:
print("=== VALIDASI AKHIR DATA CLEANING ===")

assert len(df_clean) == 183, "Jumlah record berubah!"
assert df_clean["menu_id"].is_unique, "ID menu duplikat!"
assert df_clean["category_group"].notna().all(), "Kategori belum lengkap!"
assert (df_clean["price"] > 0).all(), "Ada harga tidak valid!"

print("Jumlah record:", len(df_clean))
print("Kategori umum:", df_clean["category_group"].nunique())
print("Kategori belum dipetakan:", df_clean["category_group"].isna().sum())
print("Duplikasi ID:", df_clean["menu_id"].duplicated().sum())

print("\nValidasi struktur dan kategori berhasil.")

=== VALIDASI AKHIR DATA CLEANING ===
Jumlah record: 183
Kategori umum: 4
Kategori belum dipetakan: 0
Duplikasi ID: 0

Validasi struktur dan kategori berhasil.


In [12]:
csv_path = processed_dir / "menus_clean.csv"
xlsx_path = processed_dir / "menus_clean.xlsx"

df_clean.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

df_clean.to_excel(
    xlsx_path,
    index=False
)

print("Dataset berhasil disimpan:")
print(csv_path)
print(xlsx_path)

Dataset berhasil disimpan:
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\data\processed\menus_clean.csv
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\data\processed\menus_clean.xlsx


In [14]:
reports_dir = project_root / "reports"

In [15]:
cleaning_log = pd.DataFrame([
    {
        "step": 1,
        "process": "Standardisasi nama kolom",
        "description": "Memastikan nama kolom menggunakan snake_case",
        "result": "Struktur nama kolom konsisten"
    },
    {
        "step": 2,
        "process": "Standardisasi teks",
        "description": "Menghapus spasi awal dan akhir pada kolom teks",
        "result": "Format teks distandarkan"
    },
    {
        "step": 3,
        "process": "Validasi harga",
        "description": "Memastikan harga numerik dan positif",
        "result": "183 harga valid secara format dan rentang"
    },
    {
        "step": 4,
        "process": "Penanganan missing values",
        "description": "Mempertahankan nilai kosong pada best seller dan availability",
        "result": "Tidak dilakukan imputasi tanpa sumber"
    },
    {
        "step": 5,
        "process": "Pengelompokan kategori",
        "description": "Memetakan 20 kategori asli ke empat kategori umum",
        "result": "183 record berhasil dipetakan"
    },
    {
        "step": 6,
        "process": "Pemeriksaan duplikasi",
        "description": "Mempertahankan menu bernama sama yang berada pada kategori berbeda",
        "result": "Tidak ada record yang dihapus"
    }
])

log_path = reports_dir / "cleaning_log.csv"
log_path.parent.mkdir(parents=True, exist_ok=True)

cleaning_log.to_csv(log_path, index=False)

display(cleaning_log)
print("Cleaning log tersimpan:", log_path)

,step,process,description,result
0,1,Standardisasi nama kolom,Memastikan nama kolom menggunakan snake_case,Struktur nama kolom konsisten
1,2,Standardisasi teks,Menghapus spasi awal dan akhir pada kolom teks,Format teks distandarkan
2,3,Validasi harga,Memastikan harga numerik dan positif,183 harga valid secara format dan rentang
3,4,Penanganan missing values,Mempertahankan nilai kosong pada best seller d...,Tidak dilakukan imputasi tanpa sumber
4,5,Pengelompokan kategori,Memetakan 20 kategori asli ke empat kategori umum,183 record berhasil dipetakan
5,6,Pemeriksaan duplikasi,Mempertahankan menu bernama sama yang berada p...,Tidak ada record yang dihapus


Cleaning log tersimpan: a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\reports\cleaning_log.csv
